## Prototype 5

In [ ]:
import csv
import io
import os
import tarfile

import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import drive
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.preprocessing import MinMaxScaler

SEED = 42
START_DAY = pd.Timestamp("2010-08-11")
TRAIN_END_DAY = pd.Timestamp("2010-08-18")
END_DAY = pd.Timestamp("2010-08-25")
TOP_N_PERCENT = 0.10
EPOCHS = 50
BATCH_SIZE = 512
PATIENCE = 5
DATE_FORMAT = "%m/%d/%Y %H:%M:%S"

np.random.seed(SEED)
tf.random.set_seed(SEED)


## Locate the CERT r6.2 files


In [ ]:
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"
REQUIRED_FILES = {
    "logon.csv",
    "file.csv",
    "device.csv",
    "psychometric.csv",
    "answers.tar.bz2"
}

matching_folders = []

for current_folder, _, filenames in os.walk(DRIVE_ROOT):
    names = {name.lower() for name in filenames}
    if REQUIRED_FILES.issubset(names):
        matching_folders.append(current_folder)

if len(matching_folders) != 1:
    print("Matching folders:", matching_folders)
    raise ValueError("Exactly one Drive folder must contain all required files.")

DATA_FOLDER = matching_folders[0]
OUTPUT_FOLDER = os.path.join(DATA_FOLDER, "liu_style_outputs")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def file_path(expected_name):
    for name in os.listdir(DATA_FOLDER):
        if name.lower() == expected_name.lower():
            return os.path.join(DATA_FOLDER, name)
    raise FileNotFoundError(expected_name)


LOGON_PATH = file_path("logon.csv")
FILE_PATH = file_path("file.csv")
DEVICE_PATH = file_path("device.csv")
PSYCHOMETRIC_PATH = file_path("psychometric.csv")
ANSWER_PATH = file_path("answers.tar.bz2")

print("Data folder:", DATA_FOLDER)
print("Output folder:", OUTPUT_FOLDER)


Mounted at /content/drive
Data folder: /content/drive/MyDrive/r6.2
Output folder: /content/drive/MyDrive/r6.2/liu_style_outputs


## Load only the two-week experimental interval


In [ ]:
def load_period(path, required_columns):
    header = pd.read_csv(path, nrows=0)
    lookup = {column.strip().lower(): column for column in header.columns}
    missing = [column for column in required_columns if column not in lookup]

    if missing:
        raise ValueError(
            f"{os.path.basename(path)} is missing {missing}. "
            f"Available columns: {list(header.columns)}"
        )

    selected = [lookup[column] for column in required_columns]
    parts = []

    for chunk in pd.read_csv(
        path,
        usecols=selected,
        chunksize=500000,
        low_memory=False
    ):
        chunk.columns = [column.strip().lower() for column in chunk.columns]
        dates = pd.to_datetime(
            chunk["date"],
            format=DATE_FORMAT,
            errors="coerce"
        )
        chunk["date"] = dates
        chunk = chunk[
            (chunk["date"] >= START_DAY)
            & (chunk["date"] < END_DAY + pd.Timedelta(days=1))
        ]
        if not chunk.empty:
            parts.append(chunk)

    if not parts:
        raise ValueError(f"No rows found in {os.path.basename(path)}.")

    data = pd.concat(parts, ignore_index=True)
    data = data.dropna(subset=["date", "user"]).copy()
    data["user"] = data["user"].astype(str).str.strip().str.upper()
    data["day"] = data["date"].dt.normalize()
    data["hour"] = data["date"].dt.hour
    return data


In [ ]:
logon_events = load_period(
    LOGON_PATH,
    ["date", "user", "activity"]
)
file_events = load_period(
    FILE_PATH,
    ["date", "user", "activity"]
)
device_events = load_period(
    DEVICE_PATH,
    ["date", "user", "activity"]
)
print("Logon events:", len(logon_events))
print("File events:", len(file_events))
print("Device events:", len(device_events))
print("Active users:", len(set(logon_events["user"]) | set(file_events["user"]) | set(device_events["user"])))


Logon events: 109978
File events: 64359
Device events: 48030
Active users: 3864


## Construct Liu-style hourly user-day vectors


In [ ]:
def normalise_activity(dataframe, activity_names):
    data = dataframe.copy()
    activity_text = data["activity"].astype(str).str.strip().str.lower()
    mapped = pd.Series(index=data.index, dtype="object")

    for output_name, search_terms in activity_names.items():
        match = pd.Series(False, index=data.index)
        for term in search_terms:
            match = match | activity_text.str.contains(term, regex=False)
        mapped.loc[match & mapped.isna()] = output_name

    data["activity_group"] = mapped
    data = data.dropna(subset=["activity_group"])
    return data


def hourly_vectors(dataframe, ordered_activities, prefix):
    counts = (
        dataframe
        .groupby(["user", "day", "activity_group", "hour"])
        .size()
        .rename("count")
        .reset_index()
    )

    table = counts.pivot_table(
        index=["user", "day"],
        columns=["activity_group", "hour"],
        values="count",
        fill_value=0
    )

    columns = pd.MultiIndex.from_product(
        [ordered_activities, range(24)],
        names=["activity_group", "hour"]
    )
    table = table.reindex(columns=columns, fill_value=0)
    table.columns = [
        f"{prefix}_{activity}_{hour:02d}"
        for activity, hour in table.columns
    ]
    return table.reset_index()


In [ ]:
logon_events = normalise_activity(
    logon_events,
    {
        "logon": ["logon", "log on"],
        "logoff": ["logoff", "log off"]
    }
)
device_events = normalise_activity(
    device_events,
    {
        "disconnect": ["disconnect"],
        "connect": ["connect"]
    }
)
file_events = normalise_activity(
    file_events,
    {
        "open": ["open"],
        "write": ["write"],
        "copy": ["copy"],
        "delete": ["delete"]
    }
)
logon_profiles = hourly_vectors(
    logon_events,
    ["logon", "logoff"],
    "logon"
)
device_profiles = hourly_vectors(
    device_events,
    ["connect", "disconnect"],
    "device"
)
file_profiles = hourly_vectors(
    file_events,
    ["open", "write", "copy", "delete"],
    "file"
)
print("Logon profile shape:", logon_profiles.shape)
print("File profile shape:", file_profiles.shape)
print("Device profile shape:", device_profiles.shape)
if logon_profiles.shape[1] - 2 != 48:
    raise ValueError("Logon vector must contain 48 features.")
if file_profiles.shape[1] - 2 != 96:
    raise ValueError("File vector must contain 96 features.")
if device_profiles.shape[1] - 2 != 48:
    raise ValueError("Device vector must contain 48 features.")


Logon profile shape: (43258, 50)
File profile shape: (10114, 98)
Device profile shape: (6163, 50)


## Extract the official r6.2 malicious user-days


In [ ]:
ground_truth_records = []

with tarfile.open(ANSWER_PATH, "r:bz2") as archive:
    insiders_file = archive.extractfile("answers/insiders.csv")
    if insiders_file is None:
        raise FileNotFoundError("answers/insiders.csv")

    insiders = pd.read_csv(insiders_file, dtype=str)
    incidents = insiders[insiders["dataset"] == "6.2"]

    for _, incident in incidents.iterrows():
        insider = incident["user"].strip().upper()
        detail_name = "answers/" + incident["details"]
        detail_file = archive.extractfile(detail_name)

        if detail_file is None:
            raise FileNotFoundError(detail_name)

        text_file = io.TextIOWrapper(
            detail_file,
            encoding="utf-8",
            errors="replace",
            newline=""
        )

        for row in csv.reader(text_file):
            if len(row) < 4:
                continue

            event_date = pd.to_datetime(
                row[2],
                format=DATE_FORMAT,
                errors="coerce"
            )
            event_user = row[3].strip().upper()

            if pd.notna(event_date) and event_user == insider:
                ground_truth_records.append({
                    "user": insider,
                    "day": event_date.normalize(),
                    "scenario_id": "r6.2-scenario-" + str(incident["scenario"])
                })

ground_truth = pd.DataFrame(ground_truth_records)
ground_truth = ground_truth[
    (ground_truth["day"] >= START_DAY)
    & (ground_truth["day"] <= END_DAY)
].drop_duplicates(["user", "day", "scenario_id"])

daily_ground_truth = (
    ground_truth
    .groupby(["user", "day"], as_index=False)
    .agg(scenario_id=("scenario_id", lambda values: " | ".join(sorted(set(values)))))
)
daily_ground_truth["ground_truth_label"] = 1

print("Malicious user-days in interval:", len(daily_ground_truth))
print("Malicious users in interval:", daily_ground_truth["user"].nunique())
display(daily_ground_truth)


Malicious user-days in interval: 5
Malicious users in interval: 2


,user,day,scenario_id,ground_truth_label
0,ACM2278,2010-08-18,r6.2-scenario-1,1
1,ACM2278,2010-08-19,r6.2-scenario-1,1
2,ACM2278,2010-08-24,r6.2-scenario-1,1
3,PLJ1771,2010-08-12,r6.2-scenario-3,1
4,PLJ1771,2010-08-13,r6.2-scenario-3,1


## Train three independent autoencoders


In [ ]:
def build_autoencoder(input_size):
    first = max(input_size // 2, 2)
    second = max(input_size // 4, 2)
    bottleneck = max(input_size // 8, 2)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_size,)),
        tf.keras.layers.Dense(first, activation="relu"),
        tf.keras.layers.Dense(second, activation="relu"),
        tf.keras.layers.Dense(bottleneck, activation="relu"),
        tf.keras.layers.Dense(second, activation="relu"),
        tf.keras.layers.Dense(first, activation="relu"),
        tf.keras.layers.Dense(input_size, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="mse")
    return model


def train_stream(profiles, stream_name):
    feature_columns = [
        column for column in profiles.columns
        if column not in ["user", "day"]
    ]
    train = profiles[profiles["day"] < TRAIN_END_DAY].copy()
    detection = profiles[profiles["day"] >= TRAIN_END_DAY].copy()

    if train.empty or detection.empty:
        raise ValueError(f"{stream_name} has an empty split.")

    scaler = MinMaxScaler()
    train_x = scaler.fit_transform(train[feature_columns])
    detection_x = scaler.transform(detection[feature_columns])
    detection_x = np.clip(detection_x, 0, 1)

    model = build_autoencoder(len(feature_columns))
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True
    )
    model.fit(
        train_x,
        train_x,
        validation_split=0.15,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        callbacks=[early_stopping],
        verbose=1
    )

    train_reconstruction = model.predict(
        train_x,
        batch_size=BATCH_SIZE,
        verbose=0
    )
    detection_reconstruction = model.predict(
        detection_x,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    train_error = np.mean(
        np.square(train_x - train_reconstruction),
        axis=1
    )
    detection_error = np.mean(
        np.square(detection_x - detection_reconstruction),
        axis=1
    )

    error_mean = train_error.mean()
    error_std = train_error.std() + 1e-8
    normalised_error = (detection_error - error_mean) / error_std

    result = detection[["user", "day"]].reset_index(drop=True)
    result[f"{stream_name}_error"] = detection_error
    result[f"{stream_name}_normalised_error"] = normalised_error

    number_to_keep = max(1, int(np.ceil(len(result) * TOP_N_PERCENT)))
    selected_index = result[f"{stream_name}_normalised_error"].nlargest(
        number_to_keep
    ).index
    result[f"{stream_name}_top_score"] = 0.0
    result.loc[
        selected_index,
        f"{stream_name}_top_score"
    ] = result.loc[
        selected_index,
        f"{stream_name}_normalised_error"
    ].clip(lower=0)
    result[f"{stream_name}_flag"] = 0
    result.loc[selected_index, f"{stream_name}_flag"] = 1

    print(
        stream_name,
        "training profiles:", len(train),
        "detection profiles:", len(detection),
        "top profiles:", number_to_keep
    )

    return {
        "model": model,
        "scaler": scaler,
        "features": feature_columns,
        "results": result
    }


In [ ]:
logon_output = train_stream(logon_profiles, "logon")
file_output = train_stream(file_profiles, "file")
device_output = train_stream(device_profiles, "device")


Epoch 1/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 5s 62ms/step - loss: 0.2308 - val_loss: 0.2185
Epoch 2/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1849 - val_loss: 0.1254
Epoch 3/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0624 - val_loss: 0.0204
Epoch 4/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0126 - val_loss: 0.0092
Epoch 5/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0083 - val_loss: 0.0082
Epoch 6/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0077 - val_loss: 0.0077
Epoch 7/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0074 - val_loss: 0.0076
Epoch 8/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0073 - val_loss: 0.0075
Epoch 9/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0073 - val_loss: 0.0075
Epoch 10/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0072 - val_loss: 0.0074
Epoch 11/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0072 - val_loss: 0.0074
Epoch 12/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0071 - val_l

## Combine the top stream scores using Liu-style averaging


In [ ]:
ensemble = (
    logon_output["results"]
    .merge(file_output["results"], on=["user", "day"], how="outer")
    .merge(device_output["results"], on=["user", "day"], how="outer")
)

top_score_columns = [
    "logon_top_score",
    "file_top_score",
    "device_top_score"
]

ensemble[top_score_columns] = ensemble[top_score_columns].fillna(0.0)
ensemble["maliciousness_score"] = ensemble[top_score_columns].mean(axis=1)

ensemble_keep = max(1, int(np.ceil(len(ensemble) * TOP_N_PERCENT)))
ensemble_selected = ensemble["maliciousness_score"].nlargest(
    ensemble_keep
).index
ensemble["anomaly_flag"] = 0
ensemble.loc[ensemble_selected, "anomaly_flag"] = 1

ensemble = ensemble.merge(
    daily_ground_truth,
    on=["user", "day"],
    how="left"
)
ensemble["ground_truth_label"] = (
    ensemble["ground_truth_label"].fillna(0).astype(int)
)
ensemble["scenario_id"] = ensemble["scenario_id"].fillna("Background")

print("Detection user-days:", len(ensemble))
print("Final top-N alerts:", ensemble["anomaly_flag"].sum())
print("Malicious detection user-days:", ensemble["ground_truth_label"].sum())


Detection user-days: 23552
Final top-N alerts: 2356
Malicious detection user-days: 3


## Evaluate anomaly detection and file detection


In [ ]:
def evaluate_flag(dataframe, flag_column, score_column, model_name):
    available = dataframe.dropna(subset=[flag_column, score_column]).copy()
    y_true = available["ground_truth_label"].astype(int)
    y_pred = available[flag_column].astype(int)
    y_score = available[score_column]
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    if y_true.nunique() == 2:
        ap = average_precision_score(y_true, y_score)
        balanced = balanced_accuracy_score(y_true, y_pred)
    else:
        ap = np.nan
        balanced = np.nan

    return {
        "model": model_name,
        "rows": len(available),
        "malicious_rows": int(y_true.sum()),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced,
        "average_precision": ap,
        "alerts": int(y_pred.sum())
    }


metrics = [
    evaluate_flag(
        ensemble,
        "anomaly_flag",
        "maliciousness_score",
        "Three-autoencoder ensemble"
    )
]

for stream, output in [
    ("logon", logon_output),
    ("file", file_output),
    ("device", device_output)
]:
    stream_result = output["results"].merge(
        daily_ground_truth,
        on=["user", "day"],
        how="left"
    )
    stream_result["ground_truth_label"] = (
        stream_result["ground_truth_label"].fillna(0).astype(int)
    )
    metrics.append(
        evaluate_flag(
            stream_result,
            f"{stream}_flag",
            f"{stream}_normalised_error",
            f"{stream.title()} autoencoder"
        )
    )

metrics_table = pd.DataFrame(metrics)
display(metrics_table)


,model,rows,malicious_rows,tp,fp,fn,tn,precision,recall,f1,balanced_accuracy,average_precision,alerts
0,Three-autoencoder ensemble,23552,3,2,2354,1,21195,0.000849,0.666667,0.001696,0.783352,0.000753,2356
1,Logon autoencoder,23552,3,0,2356,3,21193,0.000000,0.000000,0.000000,0.449977,0.000395,2356
2,File autoencoder,5337,3,1,533,2,4801,0.001873,0.333333,0.003724,0.616704,0.002638,534
3,Device autoencoder,3380,3,1,337,2,3040,0.002959,0.333333,0.005865,0.616770,0.003446,338


In [ ]:
file_detection = file_output["results"].merge(
    daily_ground_truth,
    on=["user", "day"],
    how="inner"
)

file_detection["scenario_id"] = file_detection["scenario_id"].fillna("Background")

display(
    file_detection[
        [
            "user",
            "day",
            "scenario_id",
            "file_error",
            "file_normalised_error",
            "file_flag"
        ]
    ].sort_values("file_normalised_error", ascending=False)
)


,user,day,scenario_id,file_error,file_normalised_error,file_flag
1,ACM2278,2010-08-19,r6.2-scenario-1,0.010414,2.077148,1
2,ACM2278,2010-08-24,r6.2-scenario-1,0.005824,0.954622,0
0,ACM2278,2010-08-18,r6.2-scenario-1,0.000097,-0.445826,0


## Add OCEAN interpretation after technical detection


In [ ]:
psychometric = pd.read_csv(PSYCHOMETRIC_PATH)
psychometric.columns = [
    column.strip().lower()
    for column in psychometric.columns
]

user_column = next(
    column for column in [
        "user", "user_id", "userid", "employee", "employee_name"
    ]
    if column in psychometric.columns
)

trait_aliases = {
    "openness": ["openness", "o"],
    "conscientiousness": ["conscientiousness", "c"],
    "extraversion": ["extraversion", "e"],
    "agreeableness": ["agreeableness", "a"],
    "neuroticism": ["neuroticism", "n"]
}

trait_columns = {}

for trait, aliases in trait_aliases.items():
    matches = [alias for alias in aliases if alias in psychometric.columns]
    if not matches:
        raise ValueError(f"Missing psychometric trait: {trait}")
    trait_columns[trait] = matches[0]

psychometric = psychometric.rename(columns={user_column: "user"})
psychometric["user"] = (
    psychometric["user"].astype(str).str.strip().str.upper()
)

for trait, column in trait_columns.items():
    psychometric[trait] = pd.to_numeric(
        psychometric[column],
        errors="coerce"
    )

psychometric = psychometric[
    ["user"] + list(trait_columns)
].dropna().drop_duplicates("user")

training_users = set(
    logon_profiles.loc[
        logon_profiles["day"] < TRAIN_END_DAY,
        "user"
    ]
)
psychometric_training = psychometric[
    psychometric["user"].isin(training_users)
]

trait_limits = {
    trait: (
        psychometric_training[trait].quantile(0.05),
        psychometric_training[trait].quantile(0.95)
    )
    for trait in trait_columns
}


def psychology_interpretation(row):
    descriptions = []
    for trait, limits in trait_limits.items():
        value = row.get(trait, np.nan)
        if pd.isna(value):
            continue
        if value <= limits[0]:
            descriptions.append("low " + trait)
        elif value >= limits[1]:
            descriptions.append("high " + trait)
    if descriptions:
        return ", ".join(descriptions)
    return "typical OCEAN range"


final_output = ensemble.merge(
    psychometric,
    on="user",
    how="left"
)
final_output["psychology_interpretation"] = final_output.apply(
    psychology_interpretation,
    axis=1
)

final_output["final_category"] = np.where(
    final_output["anomaly_flag"] == 1,
    "Technical anomaly requiring review",
    "Ordinary technical profile"
)

print(
    "Psychometric match rate:",
    round(final_output["openness"].notna().mean(), 4)
)


Psychometric match rate: 1.0
